# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/133Haseeb/MLflyrankhaseeb/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My Rule

I created a simple baseline rule to identify pages that should be refreshed. The rule gives higher scores to pages with high impressions, low CTR, and content that has not been updated for a long time. Pages with higher scores are considered higher priority for review.

Reason Codes

Reason Code	Meaning
STALE_CONTENT	Page has not been updated for a long time

LOW_CTR	CTR is lower than expected

HIGH_IMPRESSIONS	Page receives many impressions

REFRESH_FIRST	High priority page for refresh

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
import os
import subprocess

REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git","clone","--depth","1",REPO_URL,REPO_DIR])

print("Done")

Done


In [6]:
import pandas as pd
import os

# Load dataset
df = pd.read_csv("flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv")

# -----------------------
# Hand-written Rule
# -----------------------

score = (
    (df["days_since_last_update"] > 180).astype(int)*40 +
    (df["ctr"] < 0.50).astype(int)*30 +
    (df["impressions_90d"] > 5000).astype(int)*30
)

df["baseline_score"] = score

# -----------------------
# Reason Code
# -----------------------

def reason(row):

    reasons=[]

    if row["days_since_last_update"]>180:
        reasons.append("STALE_CONTENT")

    if row["ctr"]<0.50:
        reasons.append("LOW_CTR")

    if row["impressions_90d"]>5000:
        reasons.append("HIGH_IMPRESSIONS")

    if len(reasons)==3:
        return "REFRESH_FIRST | STALE_CONTENT + LOW_CTR + HIGH_IMPRESSIONS"

    return ", ".join(reasons)

df["reason_code"]=df.apply(reason,axis=1)

# -----------------------
# Action
# -----------------------

df["action"]=df["baseline_score"].apply(
    lambda x:"Refresh Page" if x>=70 else "Monitor"
)

# -----------------------
# Rank
# -----------------------

ranked=df.sort_values("baseline_score",ascending=False)

# -----------------------
# Save CSV
# -----------------------

os.makedirs("work/outputs",exist_ok=True)

ranked.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV saved successfully!")

ranked.head(20)

CSV saved successfully!


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,baseline_score,reason_code,action
7021,content_1bfaa38ff26c,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3861.0,24672.0,...,3.75,43.33,0.0,good,page_3_5,down,-74.7,100,REFRESH_FIRST | STALE_CONTENT + LOW_CTR + HIGH...,Refresh Page
16751,content_cf56e2e2e282,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,5125.0,33705.0,...,0.84,24.11,0.0,excellent,striking,down,-85.6,100,REFRESH_FIRST | STALE_CONTENT + LOW_CTR + HIGH...,Refresh Page
16514,content_7368877ea310,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,2591.0,16498.0,...,3.66,42.99,0.0,excellent,page_3_5,down,-81.5,100,REFRESH_FIRST | STALE_CONTENT + LOW_CTR + HIGH...,Refresh Page
12045,content_c2d929d83eaa,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,4758.0,30070.0,...,0.00,40.00,0.0,good,striking,down,-62.8,100,REFRESH_FIRST | STALE_CONTENT + LOW_CTR + HIGH...,Refresh Page
21268,content_0a91db491d14,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3478.0,21948.0,...,5.13,41.76,0.0,good,striking,down,-51.8,100,REFRESH_FIRST | STALE_CONTENT + LOW_CTR + HIGH...,Refresh Page
11489,content_5feee3994adb,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,transactional,3590.0,22780.0,...,0.00,40.00,0.0,good,page_3_5,down,-89.1,100,REFRESH_FIRST | STALE_CONTENT + LOW_CTR + HIGH...,Refresh Page
24603,content_06f581dd9383,client_d4735e3a26,NaN,NaN,NaN,NaN,keyword article,NaN,809.0,5810.0,...,0.00,100.00,0.0,low,page_1,flat,NaN,70,"STALE_CONTENT, LOW_CTR",Refresh Page
6421,content_fc8cb7532683,client_d029fa3a95,0.0,0.00,LOW,0.00,keyword article,informational,3913.0,27598.0,...,0.00,100.00,0.0,low,striking,down,-62.5,70,"STALE_CONTENT, LOW_CTR",Refresh Page
15882,content_129753e3095f,client_19581e27de,10.0,0.55,MEDIUM,0.04,keyword article,transactional,NaN,NaN,...,0.00,100.00,0.0,low,page_3_5,down,-50.0,70,"STALE_CONTENT, LOW_CTR",Refresh Page
22248,content_607ea6e3f876,client_4ec9599fc2,NaN,NaN,NaN,NaN,keyword article,NaN,1308.0,9591.0,...,0.00,0.00,0.0,low,page_3_5,up,1080.0,70,"STALE_CONTENT, LOW_CTR",Refresh Page


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

| Action       | Reason Code                                | Confidence | What would make it wrong                                             |
| ------------ | ------------------------------------------ | ---------- | -------------------------------------------------------------------- |
| Refresh Page | STALE_CONTENT + LOW_CTR + HIGH_IMPRESSIONS | High       | Traffic may drop because of seasonality instead of outdated content. |
| Refresh Page | STALE_CONTENT + LOW_CTR                    | Medium     | CTR may be low because competitors improved their pages.             |
| Refresh Page | LOW_CTR + HIGH_IMPRESSIONS                 | Medium     | Page title may need improvement rather than full refresh.            |
| Refresh Page | STALE_CONTENT                              | Medium     | Content may still be accurate even if old.                           |
| Refresh Page | REFRESH_FIRST                              | High       | External events may temporarily affect performance.                  |


Similar observations were found for the remaining top-ranked pages. They were selected because they matched the baseline rule. Human review is still necessary before making any final SEO decision.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Picks

Some pages selected by the handwritten rule may not actually require refreshing. For example, a page may have low CTR because users are searching differently, or impressions may be temporarily reduced because of seasonal trends. Therefore, the baseline rule cannot understand every situation.

Leakage Check

No future information or label-derived columns were used. The rule only uses observable information available before making the refresh decision, such as impressions, CTR, and days since the last update.

## Self-check

Before you submit, confirm each line honestly:

- [ Checked] Every section above is filled — markdown thinking AND the code that backs it
- [ Checked] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ Checked ] No client names, URLs, or private queries anywhere
- [ Checked ] My claims use careful words: observed, measured, directional, decision-support
- [ Checked ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.